# Random search algorithm

In [1]:
METHOD = "RANDOM"

In [2]:
import io
import os
import random
import torch
import json
import contextlib
from ultralytics import YOLO 
from mylib import simsettings
from mylib import myutils
from mylib import simtools
from mylib import yolo_patch_softmax as _
from constants import OBS_SCALE, CELL_SIDE, MAP_RESOLUTION, DISPLAY_STEP
from constants import AGENT_HEIGHT, AGENT_RADIUS 
from constants import MAX_ITER_COEF, CONFIDENCE_THRESHOLD, LOCATION_ERROR_THRESHOLD, PSEUDO_COUNT_THRESHOLD
from constants import ACTIONS   
from constants import NUM_EPOCHS
from dotenv import load_dotenv
load_dotenv()

import habitat_sim
import habitat_sim.nav as nav
from habitat.utils.visualizations import maps
from habitat_sim.utils import common as utils

# Reload imported modules
%load_ext autoreload
%autoreload 2

# Load YOLO model
yolo_model = YOLO("yolo11x.pt")  

# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
with contextlib.redirect_stdout(io.StringIO()):
    yolo_model = yolo_model.to(device)

In [4]:
# Load the JSON file for simulation
with open('simulation-data/simulations.json', 'r') as f:
    simulations = json.load(f)

# Simulation configuration (index selects the simulation)
simulation = simulations[23]

# Access its fields
SCENE = simulation["scene"]
TARGET_OBJECT = simulation["target_object"]
TARGET_OBJECT_ID = simulation["target_object_id"]
REAL_TARGET_LOCATION = simulation["target_object_location"]
INDEX = simulation["index"]

# Load the JSON file for RGB camera intrinsics
with open('simulation-data/camera-intrinsics.json', 'r') as f:
    intrinsics = json.load(f)

# Simulator configuration
dataset_config_file = os.path.join(os.getenv("AI2THOR_DATA"), "ai2thor-hab.scene_dataset_config.json")
sim_settings = {
    "seed": 1,
    "dataset": dataset_config_file,  # Scene dataset
    "scene": SCENE,  # Scene path
    "width": 1024,  # Spatial resolution of the observations
    "height": int(1024*OBS_SCALE),
    "default_agent": 0,
    "sensor_height": AGENT_HEIGHT,  # Height of sensors in meters
    "color_sensor": True,  # RGB sensor
    "depth_sensor": True,  # Depth sensor
    "enable_physics": False,  # kinematics only
}
# Initialize the simulator
cfg = simsettings.make_cfg(sim_settings)
sim = habitat_sim.Simulator(cfg)

In [5]:
# Get the root node of the active scene graph
scene_root = sim.get_active_scene_graph().get_root_node()
scene_bb = scene_root.cumulative_bb
scene_dims = scene_bb.size()

# Define navmesh settings
navmesh_settings = habitat_sim.NavMeshSettings()
navmesh_settings.agent_height = AGENT_HEIGHT
navmesh_settings.agent_radius = AGENT_RADIUS
navmesh_settings.agent_max_climb = 0.2
navmesh_settings.agent_max_slope = 45.0
navmesh_settings.include_static_objects = True  # Include static objects in the navmesh computation

# Recompute the navmesh for the current scene
sim.recompute_navmesh(sim.pathfinder, navmesh_settings)

# Generate the top-down map --> 1 cm per pixel
topdown_map = maps.get_topdown_map(sim.pathfinder, height=0, meters_per_pixel=MAP_RESOLUTION, draw_border=True)
topdown_map = myutils.map_to_rgb(topdown_map)
topdown_map = myutils.add_axis_to_map(topdown_map)
topdown_map = myutils.retain_largest_white_chunk(topdown_map)
topdown_resolution = topdown_map.shape[:2]

# Generate the coarse map --> 30 cm per pixel (robot has radius 15 cm)
grid_map = maps.get_topdown_map(sim.pathfinder, height=0, meters_per_pixel=CELL_SIDE, draw_border=False)
grid_map = myutils.map_to_rgb(grid_map)
grid_map = myutils.retain_largest_white_chunk(grid_map)
grid_resolution = grid_map.shape[:2]

# Free positions
grid_free_cells, map_free_cells, world_free_coords = [], [], []

for x_g in range(grid_resolution[0]):   
    for y_g in range(grid_resolution[1]):
        if grid_map[x_g, y_g, 0] == 255: # White pixel
            # Convert to real world position and map position
            real_wrld_z, real_wrld_x = maps.from_grid(x_g, y_g, grid_resolution, pathfinder=sim.pathfinder)
            map_x, map_y = maps.to_grid(real_wrld_z, real_wrld_x, topdown_resolution, pathfinder=sim.pathfinder)

            # Check if the position is navigable and not occupied by an object
            if not sim.pathfinder.is_navigable([real_wrld_x, 0.0, real_wrld_z]) or not topdown_map[map_x, map_y, 0] == 255: # White pixel
                grid_map[x_g, y_g, :] = [128, 128, 128] # Grey pixel

# Filter map again to retain only the largest white chunk              
grid_map = myutils.retain_largest_white_chunk(grid_map)

for x_g in range(grid_resolution[0]):
    for y_g in range(grid_resolution[1]):
        if grid_map[x_g, y_g, 0] == 255: # White pixel
            # Convert to real world position and map position
            real_wrld_z, real_wrld_x = maps.from_grid(x_g, y_g, grid_resolution, pathfinder=sim.pathfinder)
            map_x, map_y = maps.to_grid(real_wrld_z, real_wrld_x, topdown_resolution, pathfinder=sim.pathfinder)

            # Check if the position is navigable and not occupied by an object
            if sim.pathfinder.is_navigable([real_wrld_x, 0.0, real_wrld_z]) and topdown_map[map_x, map_y, 0] == 255: # White pixel
                world_free_coords.append([real_wrld_x, 0.0, real_wrld_z])
                map_free_cells.append([map_x, map_y])
                grid_free_cells.append([x_g, y_g]) # Add grid position

# Count the number of occupiable positions
num_free_cells = len(grid_free_cells)

In [ ]:
# starting_grid_positions = []
# starting_orientations = []

# for _ in range(NUM_EPOCHS):
#     # Randomly select a free cell
#     starting_grid_positions.append(random.choice(grid_free_cells))
#     starting_orientations.append(random.choice([0, 90, 180, 270])) # Random orientation in degrees

# # Print it like it was a list
# print("starting_grid_positions = [", end="")
# for pos in starting_grid_positions:
#     print(f"{pos}, ", end="")
# print("]")

# print("starting_orientations = [", end="")
# for ori in starting_orientations:
#     print(f"{ori}, ", end="")
# print("]")

starting_grid_positions = [[9, 36], [17, 27], [33, 27], [1, 44], [17, 10], [26, 18], [21, 11], [11, 22], [28, 13], [23, 31], [15, 42], [28, 28], [14, 41], [5, 44], [10, 41], [3, 40], [14, 39], [16, 2], [19, 40], [19, 41], [30, 8], [17, 5], [24, 28], [13, 34], [8, 38], [14, 28], [31, 24], [25, 28], [30, 5], [11, 39], [32, 8], [15, 14], [17, 4], [30, 3], [27, 23], [13, 32], [9, 14], [22, 12], [9, 15], [16, 16], [15, 26], [11, 34], [11, 44], [17, 29], [5, 41], [8, 45], [16, 11], [20, 17], [22, 15], [19, 3], [15, 44], [32, 16], [10, 26], [9, 31], [26, 11], [17, 9], [27, 13], [12, 23], [18, 23], [9, 43], [28, 32], [8, 15], [14, 45], [19, 33], [11, 29], [28, 33], [9, 26], [6, 42], [27, 27], [14, 14], [28, 26], [16, 47], [11, 36], [10, 18], [15, 22], [15, 28], [28, 24], [27, 24], [19, 13], [21, 27], [28, 23], [9, 42], [19, 33], [23, 23], [29, 29], [18, 12], [13, 44], [10, 40], [10, 16], [21, 23], [15, 44], [12, 37], [17, 19], [30, 17], [2, 36], [11, 34], [20, 26], [18, 41], [25, 24], [32, 5],

In [6]:
starting_grid_positions = [[18, 15], [23, 24], [18, 2], [18, 13], [17, 46], [18, 2], [12, 31], [24, 18], [19, 6], [14, 27], [24, 29], [15, 17], [17, 22], [31, 16], [25, 12], [15, 34], [15, 23], [4, 42], [15, 42], [12, 14], [10, 23], [15, 4], [16, 15], [18, 42], [20, 31], [31, 27], [20, 28], [16, 27], [16, 33], [18, 22], [25, 28], [12, 22], [33, 4], [7, 40], [7, 40], [21, 28], [9, 14], [21, 11], [23, 8], [20, 34], [30, 31], [26, 11], [25, 28], [13, 15], [6, 42], [4, 42], [10, 44], [23, 29], [30, 17], [30, 24], [9, 45], [27, 23], [23, 8], [30, 13], [12, 30], [1, 44], [5, 40], [12, 45], [19, 30], [33, 31], [6, 41], [17, 24], [11, 17], [19, 14], [8, 25], [6, 39], [13, 33], [22, 8], [9, 24], [18, 15], [12, 38], [33, 4], [33, 10], [32, 31], [23, 25], [11, 14], [16, 24], [29, 10], [28, 31], [15, 28], [24, 12], [22, 23], [10, 41], [23, 26], [19, 40], [3, 36], [9, 14], [13, 37], [11, 41], [29, 12], [16, 45], [9, 23], [13, 41], [8, 28], [6, 46], [4, 39], [9, 16], [23, 23], [16, 40], [20, 19], [16, 29], [15, 43], [31, 26], [25, 17], [23, 11], [24, 14], [3, 45], [19, 6], [21, 24], [14, 41], [27, 24], [10, 27], [25, 26], [16, 6], [8, 43], [3, 39], [14, 45], [27, 32], [12, 28], [33, 20], [19, 20], [22, 27], [31, 22], [12, 44], [26, 31], [26, 26], [12, 45], [24, 16], [27, 24], [28, 32], [27, 29], [8, 43], [9, 25], [29, 3], [31, 31], [25, 29], [28, 28], [29, 30], [32, 27], [10, 23], [17, 32], [13, 37], [12, 40], [33, 31], [32, 34], [18, 45], [5, 40], [28, 27], [30, 18], [11, 34], [27, 11], [23, 28], [29, 28], [5, 45], [30, 33], [24, 24], [9, 45], [29, 14], [30, 17], [12, 17], [9, 29], [14, 40], [5, 39], [20, 17], [22, 28], [14, 39], [27, 24], [25, 28], [14, 15], [14, 41], [12, 36], [14, 16], [13, 29], [25, 31], [25, 27], [14, 32], [28, 12], [25, 16], [26, 10], [33, 10], [24, 30], [20, 28], [1, 45], [20, 27], [14, 41], [30, 14], [10, 27], [5, 36], [9, 36], [20, 31], [11, 44], [29, 29], [4, 39], [11, 38], [7, 46], [19, 45], [17, 15], [11, 21], [14, 27], [14, 14], ]
starting_orientations = [270, 90, 270, 270, 180, 0, 0, 0, 270, 90, 180, 180, 270, 270, 180, 0, 90, 180, 270, 180, 270, 0, 90, 180, 90, 180, 180, 0, 90, 180, 90, 90, 0, 180, 90, 0, 270, 270, 0, 0, 180, 90, 180, 90, 270, 180, 90, 0, 0, 270, 0, 90, 180, 90, 90, 180, 90, 270, 180, 180, 180, 0, 180, 90, 90, 270, 270, 180, 90, 180, 0, 90, 0, 0, 180, 270, 90, 0, 0, 90, 180, 0, 270, 270, 180, 0, 90, 90, 0, 270, 180, 90, 90, 90, 90, 270, 0, 0, 0, 0, 270, 270, 0, 180, 180, 0, 0, 180, 180, 90, 0, 0, 270, 90, 270, 90, 180, 0, 270, 90, 0, 0, 270, 0, 90, 0, 270, 270, 90, 0, 0, 270, 270, 0, 90, 90, 90, 90, 90, 180, 270, 0, 270, 0, 90, 90, 0, 180, 90, 0, 90, 90, 0, 180, 0, 90, 180, 90, 270, 90, 270, 180, 90, 0, 0, 270, 90, 180, 270, 180, 90, 0, 0, 90, 270, 90, 180, 90, 270, 180, 270, 180, 270, 180, 90, 0, 0, 180, 180, 0, 270, 180, 270, 180, 0, 180, 180, 270, 0, 0, ]

In [7]:
# Epochs metrics
num_actions_epochs = []
travelled_distance_epochs = []
success_epochs = []
location_error_epochs = []

In [8]:
for index, (GRID_POSITION, AGENT_YAW) in enumerate(zip(starting_grid_positions, starting_orientations)):

    # Initialize an agent
    agent = sim.initialize_agent(sim_settings["default_agent"])

    # Sample a random position (within the possible ones)
    grid_position = GRID_POSITION
    idx = grid_free_cells.index(grid_position)
    world_position = world_free_coords[idx]
    map_position = map_free_cells[idx]

    # Sample a random yaw rotation
    agent_yaw = AGENT_YAW  # in degrees
    agent_quart = myutils.yaw_to_quaternion(agent_yaw)

    # Set agent state
    agent_state = habitat_sim.AgentState()
    agent_state.position = world_position
    agent_state.rotation = agent_quart
    agent.set_state(agent_state)

    # Compute agent radius in both maps
    min_bounds, max_bounds = sim.pathfinder.get_bounds()
    x_dim = max_bounds[0] - min_bounds[0]
    topdown_radius = (AGENT_RADIUS / x_dim * topdown_resolution[0])
    grid_radius = (AGENT_RADIUS / x_dim * grid_resolution[0])

    # Get initial agent position tuple and radius tuple
    agent_radius = (topdown_radius, grid_radius)
    agent_positions = (map_position, grid_position)

    # Get initial observations and maps
    observations = sim.get_sensor_observations(0)
    rgb = observations["color_sensor"]
    depth = observations["depth_sensor"]

    # Display the initial simulation state (maps + observations)
    print(f"\n\nSimulation {index}/{len(starting_grid_positions)}")
    #simtools.display_sim_state(rgb, depth, topdown_map, grid_map, agent_positions, agent_radius, agent_yaw)

    # Metrics initialization
    target_found = False
    num_actions = 0
    travelled_distance = 0.0
    location_error = float("inf")

    # Simulation parameters
    ACTIONS = ACTIONS # default 
    MAX_ITER = int(num_free_cells * MAX_ITER_COEF)  # Maximum number of actions to perform

    # Main simulation loop
    while (num_actions < MAX_ITER) and (not target_found):

        # Select an action randomly
        action = random.choice(list(ACTIONS))
        num_actions += 1

        # Check if the action is valid
        if not simtools.is_action_valid(action, grid_position, agent_yaw, grid_free_cells):
            continue 
        
        # Perform the action and update agent state
        grid_position, agent_yaw = simtools.perform_action(action, grid_position, agent_yaw)
        
        # Update the agent position and orientation
        idx = grid_free_cells.index(grid_position)
        map_position, world_position = map_free_cells[idx], world_free_coords[idx]
        agent_quart = myutils.yaw_to_quaternion(agent_yaw)
        agent_positions = (map_position, grid_position)

        # Compute travelled distance
        travelled_distance += simtools.compute_travelled_distance(agent_state.position, world_position)

        # Set agent state
        agent_state.position = world_position
        agent_state.rotation = agent_quart
        agent.set_state(agent_state)

        # Get observations
        obs = sim.get_sensor_observations(0)
        rgb, depth = obs["color_sensor"], obs["depth_sensor"]

        # YOLO Prediction
        results = yolo_model.predict(source=rgb[:,:,:3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
        detections = simtools.parse_yolo_detections(results)
        simtools.merge_rgb_yolo_outputs(rgb, detections)
        target_found, target_bbox = simtools.was_target_found(TARGET_OBJECT_ID, detections, CONFIDENCE_THRESHOLD)

    # Target found or not
    if target_found:
        # Real world position 
        center_x, center_y = simtools.get_box_center(target_bbox)
        depth_value = depth[center_y, center_x]
        target_object_position = simtools.compute_real_world_position_from_pixel(agent_state.position, agent_state.rotation, depth_value, center_x, center_y, intrinsics)

        # Location_error to the target location
        location_error = simtools.compute_location_error(target_object_position, REAL_TARGET_LOCATION)

        # Convert to 2D coordinates
        target_map_position, target_grid_position = simtools.get_2d_coords(target_object_position, topdown_resolution, grid_resolution, sim.pathfinder)
        target_real_map_position, target_real_grid_position = simtools.get_2d_coords(REAL_TARGET_LOCATION, topdown_resolution, grid_resolution, sim.pathfinder)
        target_2d_coords = (target_map_position, target_grid_position)
        target_real_2d_coords = (target_real_map_position, target_real_grid_position)
        target_coords = (target_real_2d_coords, target_2d_coords)

        # Check if location is valid
        if location_error > LOCATION_ERROR_THRESHOLD:
            print(f"\nTarget object <{TARGET_OBJECT}> found after {num_actions} actions, but location error {location_error:.3f} m exceeds threshold {LOCATION_ERROR_THRESHOLD} m. FAILURE!")
            target_found = False
            location_error = float("inf")
        else:
            print(f"\nTarget object <{TARGET_OBJECT}> found after {num_actions} actions!")

        print(f"Found location: {target_object_position}")
        print(f"Real location: {REAL_TARGET_LOCATION}")
        #simtools.display_sim_observations(rgb, depth)
        #simtools.display_topdown_maps_with_target(topdown_map, grid_map, agent_positions, agent_radius, agent_yaw, target_coords)
    else:
        print(f"\nTarget object <{TARGET_OBJECT}> not found after {MAX_ITER} actions!")
        print(f"Real location: {REAL_TARGET_LOCATION}")

    # Display simulation metrics
    print(f"Number of actions: {num_actions}")
    print(f"Travelled distance: {travelled_distance:.2f} m")
    print(f"Computed location error: {location_error:.3f} m")

    # Store metrics
    num_actions_epochs.append(num_actions)
    travelled_distance_epochs.append(travelled_distance)
    success_epochs.append(target_found)
    location_error_epochs.append(location_error)




Simulation 0/200

Target object <toilet> not found after 589 actions!
Real location: [-14.301100597105403, 0.001768748760223371, 8.834680106408461]
Number of actions: 589
Travelled distance: 48.47 m
Computed location error: inf m


Simulation 1/200

Target object <toilet> not found after 589 actions!
Real location: [-14.301100597105403, 0.001768748760223371, 8.834680106408461]
Number of actions: 589
Travelled distance: 55.32 m
Computed location error: inf m


Simulation 2/200

Target object <toilet> not found after 589 actions!
Real location: [-14.301100597105403, 0.001768748760223371, 8.834680106408461]
Number of actions: 589
Travelled distance: 46.31 m
Computed location error: inf m


Simulation 3/200

Target object <toilet> not found after 589 actions!
Real location: [-14.301100597105403, 0.001768748760223371, 8.834680106408461]
Number of actions: 589
Travelled distance: 53.19 m
Computed location error: inf m


Simulation 4/200

Target object <toilet> not found after 589 actions!


In [9]:
import numpy as np

# Pre-process
num_actions_epochs = np.array(num_actions_epochs)
travelled_distance_epochs = np.array(travelled_distance_epochs)
success_epochs = np.array(success_epochs)
location_error_epochs = np.array(location_error_epochs)

# Counting
num_total_epochs = len(num_actions_epochs)
num_success_epochs = np.sum(success_epochs)

# Metrics on all runs
total_avg_num_actions = np.sum(num_actions_epochs) / num_total_epochs
total_avg_tavelled_distance = np.sum(travelled_distance_epochs) / num_total_epochs
success_rate = num_success_epochs / num_total_epochs * 100

# Metrics on successful runs
success_avg_travelled_distance = np.sum(travelled_distance_epochs[success_epochs]) / num_success_epochs
success_avg_num_actions = np.sum(num_actions_epochs[success_epochs]) / num_success_epochs
success_avg_location_error = np.sum(location_error_epochs[success_epochs]) / num_success_epochs

#Print final metrics
print("\n\nMETRICS:\n")
print(f"Total number of epochs: {num_total_epochs}")
print(f"Number of successful epochs: {num_success_epochs}")
print(f"Success rate: {success_rate:.2f}%")
print(f"Average number of actions (epochs): {total_avg_num_actions:.2f}")
print(f"Average travelled distance (epochs): {total_avg_tavelled_distance:.2f} m")
print(f"Average success rate (epochs): {success_rate:.2f}%")
print(f"\nAverage number of actions (successful epochs): {success_avg_num_actions:.2f}")
print(f"Average travelled distance (successful epochs): {success_avg_travelled_distance:.2f} m")
print(f"Average location error (successful epochs): {success_avg_location_error:.3f} m")

# Number of actions max, min and std
print(f"\nMax number of actions (epochs): {np.max(num_actions_epochs)}")
print(f"Min number of actions (epochs): {np.min(num_actions_epochs)}")
print(f"Std number of actions (epochs): {np.std(num_actions_epochs)}")    



METRICS:

Total number of epochs: 200
Number of successful epochs: 20
Success rate: 10.00%
Average number of actions (epochs): 546.25
Average travelled distance (epochs): 46.42 m
Average success rate (epochs): 10.00%

Average number of actions (successful epochs): 161.50
Average travelled distance (successful epochs): 12.93 m
Average location error (successful epochs): 0.162 m

Max number of actions (epochs): 589
Min number of actions (epochs): 2
Std number of actions (epochs): 141.7956892856761


In [10]:
metrics_file = 'results/metrics.json'

# Load existing metrics if the file exists, otherwise start with an empty list
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        existing_metrics = json.load(f)
else:
    existing_metrics = []

# New metrics entry (convert numpy types to native Python types)
def to_python_type(val):
    if hasattr(val, "item"):
        return val.item()
    return val

new_metrics = {
    "simulation_index": to_python_type(INDEX),
    "scene": SCENE,
    "target_object": TARGET_OBJECT,
    "search_method": METHOD,
    "confidence_threshold": to_python_type(CONFIDENCE_THRESHOLD),
    "max_iter_coefficient": to_python_type(MAX_ITER_COEF),
    "location_error_threshold": to_python_type(LOCATION_ERROR_THRESHOLD),
    "pseudo_count_threshold": to_python_type(PSEUDO_COUNT_THRESHOLD),
    "num_total_epochs": to_python_type(num_total_epochs),
    "num_success_epochs": to_python_type(num_success_epochs),
    "success_rate": to_python_type(success_rate),
    "total_avg_num_actions": to_python_type(total_avg_num_actions),
    "total_avg_tavelled_distance": to_python_type(total_avg_tavelled_distance),
    "success_avg_num_actions": to_python_type(success_avg_num_actions),
    "success_avg_travelled_distance": to_python_type(success_avg_travelled_distance),
    "success_avg_location_error": to_python_type(success_avg_location_error),
}

# Check for duplicate (by simulation_index and search_method)
duplicate_exists = any(
    entry["simulation_index"] == new_metrics["simulation_index"] and
    entry["search_method"] == new_metrics["search_method"] and
    entry["confidence_threshold"] == new_metrics["confidence_threshold"] and 
    entry["max_iter_coefficient"] == new_metrics["max_iter_coefficient"] and
    entry["location_error_threshold"] == new_metrics["location_error_threshold"] and
    entry["pseudo_count_threshold"] == new_metrics["pseudo_count_threshold"]
    for entry in existing_metrics
)

if not duplicate_exists:
    existing_metrics.append(new_metrics)
else:
    print(f"Metrics for simulation_index {new_metrics['simulation_index']} and search_method '{new_metrics['search_method']}' already exist. Skipping append.")

# Save updated metrics list
with open(metrics_file, 'w') as f:
    json.dump(existing_metrics, f, indent=4)

print(f"\nMetrics saved to {metrics_file}")


Metrics saved to results/metrics.json


In [11]:
sim.close()  # Close the simulator